In [1]:
import os
import json
import pandas as pd
import traceback

In [ ]:
import os, json
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0.7
)

def generate_mcqs(topic: str, num_questions: int = 3, difficulty: str = "medium"):
    prompt = f"""
    Generate {num_questions} multiple choice questions about "{topic}".
    Difficulty level: {difficulty}.

    Return ONLY valid JSON, no extra text, in this exact format:
    [
      {{
        "question": "...",
        "options": {{"A": "...", "B": "...", "C": "...", "D": "..."}},
        "correct_answer": "A"
      }}
    ]
    """
    response = llm.invoke(prompt)
    # Clean up in case Gemini wraps it in ```json ... ```
    text = response.content.strip().removeprefix("```json").removesuffix("```").strip()
    return json.loads(text)

mcqs = generate_mcqs("python programming", num_questions=3, difficulty="medium")
for q in mcqs:
    print(q["question"])
    for k, v in q["options"].items():
        print(f"  {k}. {v}")
    print("Answer:", q["correct_answer"], "\n")

In [2]:
import os
import json
import PyPDF2
import traceback

from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

In [3]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

True

In [4]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0.7
)

In [5]:
response = llm.invoke("what is 6*45?")
print(response.content)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


To calculate 6 * 45:

You can break it down:
6 * 40 = 240
6 * 5 = 30
240 + 30 = 270

So, 6 * 45 = **270**.


In [6]:
mcq_prompt = PromptTemplate(
    input_variables=["content", "num_questions", "difficulty", "tone", "mcq_type"],
    template="""
    Generate {num_questions} multiple choice questions about "{content}".
    Difficulty level: {difficulty} and also with the tone of  {tone} and this {mcq_type} type of mcqs.

    Return ONLY valid JSON, no extra text, in this exact format:
    [
      {{
        "question": "...",
        "options": {{"A": "...", "B": "...", "C": "...", "D": "..."}},
        "correct_answer": "A"
      }}
    ]
    """
)

In [7]:
chain = mcq_prompt | llm
chain_response = chain.invoke({
    "content": "agents in AI",
    "num_questions": 3,
    "difficulty": "medium",
    "tone": "exam type",
    "mcq_type": "single correct answer"
})

In [8]:
cleaned = chain_response.content.strip()
cleaned = cleaned.removeprefix("```json").removesuffix("```").strip()

mcqs = json.loads(cleaned)
print(mcqs)

[{'question': 'Which of the following best describes the fundamental characteristic of an intelligent agent in Artificial Intelligence?', 'options': {'A': 'It perceives its environment through sensors and acts upon that environment through actuators.', 'B': 'It is exclusively a software program designed to perform computations.', 'C': 'It possesses the ability to learn and adapt to dynamic changes in its environment.', 'D': 'It operates without any predefined goals or objectives, relying purely on reactive behavior.'}, 'correct_answer': 'A'}, {'question': "In the context of designing and analyzing AI agents, the PEAS (Performance, Environment, Actuators, Sensors) description is utilized. Which element of PEAS primarily defines the objective criteria for evaluating the success of an agent's actions?", 'options': {'A': 'Actuators', 'B': 'Environment', 'C': 'Performance', 'D': 'Sensors'}, 'correct_answer': 'C'}, {'question': 'According to the standard definition in Artificial Intelligence

In [10]:
def get_content(input_type: str, source: str) -> str:
    if input_type == "topic":
        return source
    elif input_type == "text":
        return source
    elif input_type == "file":
        if source.endswith(".txt"):
            with open(source, "r") as f:
                return f.read()
        elif source.endswith(".pdf"):
            reader = PyPDF2.PdfReader(source)
            result = ""
            for page in reader.pages:
                result += page.extract_text()
            return result
    else:
        raise ValueError("Invalid input_type")

In [12]:
print(get_content("topic", "photosynthesis"))
print(get_content("text", "Some pasted paragraph here."))
print(get_content("file", "C:\\Users\\verma\\mcq_gen\\data.txt"))   # only if you have a sample txt file

photosynthesis
Some pasted paragraph here.
The term machine learning was coined in 1959 by Arthur Samuel, an IBM employee and pioneer in the field of computer gaming and artificial intelligence.[5][6] The synonym self-teaching computers was also used during this time period.[7][8]

The earliest machine learning program was introduced in the 1950s, when Samuel invented a computer program that calculated the chance of winning in checkers for each side, but the history of machine learning is rooted in decades of efforts to study human cognitive processes.[9] In 1949, Canadian psychologist Donald Hebb published the book The Organization of Behavior, in which he introduced a theoretical neural structure formed by certain interactions among nerve cells.[10] The Hebbian theory of neuron interaction set the groundwork for how many machine learning algorithms work, with connected artificial neurons changing the strength of their connections based on data.[9] Other researchers who have studied h

In [13]:
def generate_mcqs(input_type,source, num_questions, difficulty, tone, mcq_type):
    content = get_content(input_type, source)
    chain = mcq_prompt | llm
    chain_response = chain.invoke({
        "content": content,
        "num_questions": num_questions,
        "difficulty": difficulty,
        "tone": tone,
        "mcq_type": mcq_type    
    })

    cleaned = chain_response.content.strip()
    cleaned = cleaned.removeprefix("```json").removesuffix("```").strip()
    mcqs = json.loads(cleaned)  
    return mcqs

In [14]:
result = generate_mcqs(
    input_type="topic",
    source="Newton's laws of motion",
    num_questions=3,
    difficulty="medium",
    tone="exam type",
    mcq_type="single correct answer"
)

for q in result:
    print(q["question"])
    for k, v in q["options"].items():
        print(f"  {k}. {v}")
    print("Answer:", q["correct_answer"], "\n")

A 10 kg object is subjected to a net force of 50 N. If the mass of the object is doubled to 20 kg while the net force remains constant at 50 N, what will happen to the object's acceleration?
  A. The acceleration will double.
  B. The acceleration will remain the same.
  C. The acceleration will be halved.
  D. The acceleration will quadruple.
Answer: C 

An astronaut in deep space, far from any gravitational influences, throws a wrench forward. According to Newton's Third Law of Motion, what will be the immediate consequence for the astronaut?
  A. The astronaut will move forward at the same speed as the wrench.
  B. The astronaut will remain stationary because there is no friction in space.
  C. The astronaut will move backward due to an equal and opposite reaction force.
  D. The astronaut will move forward, but at a slower speed than the wrench.
Answer: C 

A hockey puck slides across a frictionless ice surface at a constant velocity. If no external forces are applied to the puck, 

In [16]:
quiz = pd.DataFrame(result)